## Reproducibility & AI-assistance disclosure

Part of **`dislocation-speech-translation-fr-en`**, companion to the M2 thesis *Dislocation under Translation: A Corpus Study of Whisper and XLS-R on Spontaneous Spoken French* (Zoé de Vries, Université Paris Cité, 2026; supervised by N. Ballier).

**AI-assistance disclosure.** The executable code is the author's. The explanatory **comments and markdown** were drafted with the help of a large language model (LLM) and reviewed by the author; the LLM changed no executable logic.

**Pipeline order:** `01_whisper` → `02_xlsr` → `03_align` → `04_score`.


> **Provenance note.** The committed `data/dislocation_translations.csv` is the hand-verified alignment used in the thesis. This notebook documents the alignment *method*; re-running it over `segment_translations.csv` reproduces the spans but may differ from the committed file on some dislocation-level outputs (notably `whisper_en_cont`), which were finalised in a spreadsheet workflow. Treat the committed CSV as canonical.

# Alignement dislocations ↔ sorties modèles

Pour chaque dislocation prototypique annotée dans `dislocations_musso_proto_FINAL.xlsx`,
ce notebook retrouve le(s) segment(s) Whisper correspondant(s) dans `translations_xlsr_aligned.csv`
et concatène les sorties FR/EN des deux modèles (Whisper, XLS-R) sur cette fenêtre.

**Méthode.** Range join temporel large autour du timestamp CFPP, suivi d'un fuzzy match
textuel (mots de contenu) entre la citation et `whisper_fr`. Extension à droite tant que
les segments suivants contiennent encore des mots de la citation (limite : 3 segments).

**Sortie.** Un CSV prêt pour le scoring sacrebleu (références CALQUE + IDIOMATIC déjà incluses).

## 1. Configuration

**Si exécution dans Colab** : choisir l'une des deux options ci-dessous (mount Drive ou upload direct).
**Si exécution locale** : indiquer les chemins absolus dans la cellule suivante.

In [ ]:
# Path config (edit for local/Colab). Inputs: the dislocation test-set CSV + the segment-level model
# outputs. Output: one row per dislocation with model outputs concatenated over its aligned span.

# --- OPTION A : Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')

# --- OPTION B : upload direct dans la session Colab ---
# from google.colab import files
# uploaded = files.upload()  # déposer dislocations_musso_proto_FINAL.xlsx + translations_xlsr_aligned.csv

In [ ]:
# Imports.

# Chemins des fichiers — à adapter
DISLOC_PATH       = 'data/dislocation_test_set.csv'
TRANSLATIONS_PATH = 'data/segment_translations.csv'
OUTPUT_PATH       = 'data/dislocation_translations.csv'

## 2. Imports

In [ ]:
# Load the 79 annotated dislocations (test set) and the 874-row segment-level outputs.

import pandas as pd
import re

## 3. Chargement des fichiers

In [ ]:
# Helpers: normalise (lowercase, strip punctuation) and content_overlap_ratio for the fuzzy match.

disloc = pd.read_csv(DISLOC_PATH)
csv    = pd.read_csv(TRANSLATIONS_PATH)

print(f'Dislocations chargées : {len(disloc)} lignes')
print(f'Segments CSV chargés  : {len(csv)} lignes')
print(f'\nColonnes dislocations : {list(disloc.columns)}')
print(f'Colonnes CSV          : {list(csv.columns)}')

## 4. Fonctions utilitaires

Le fuzzy match repose sur le **chevauchement des mots de contenu** entre la citation
annotée et la transcription Whisper d'un segment. Les *mots de contenu* sont les mots
de longueur > 1 qui ne figurent pas dans la stoplist ci-dessous. La stoplist est
volontairement basique et entièrement modifiable.

In [ ]:
# Align one dislocation: -5 s / +60 s window around the CFPP timestamp, pick the highest
# content-word-overlap segment as anchor, extend right up to 3 segments while overlap persists.

STOPWORDS = set('''
le la les un une des de du d à a au aux et ou est c'est ça je tu il elle on nous vous
ils elles que qui quoi pas ne se s en y me te dans pour
'''.split())

def normalize(text):
    """Minuscules, suppression de la ponctuation, normalisation des espaces."""
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def content_overlap_ratio(citation, segment_text):
    """Fraction des mots de contenu de la citation présents dans le segment."""
    content_words = [w for w in normalize(citation).split()
                     if w not in STOPWORDS and len(w) > 1]
    if not content_words:
        return 0.0
    seg_words = set(normalize(segment_text).split())
    matched = sum(1 for w in content_words if w in seg_words)
    return matched / len(content_words)

## 5. Algorithme d'alignement

Pour chaque dislocation :

1. Construire une **fenêtre temporelle** de `−5s` à `+60s` autour du timestamp CFPP.
2. Pour chaque segment de la fenêtre, calculer le ratio de chevauchement textuel avec la citation.
3. **Segment d'ancrage** = celui qui maximise ce ratio.
4. **Extension à droite** : ajouter jusqu'à `max_extension` segments suivants tant qu'ils contiennent eux aussi des mots de la citation.

La fenêtre asymétrique (−5s à +60s) reflète l'observation empirique : les timestamps
CFPP marquent typiquement le **début du tour de parole**, qui précède la dislocation
elle-même.

In [ ]:
# Run the alignment over all dislocations; no-match rows get n_segs=0 and empty outputs.
# Output columns use whisper_en_per_seg / whisper_en_cont / xlsr_en (no _concat suffix).

def align_dislocation(row, csv_df, window_back=5, window_fwd=60, max_extension=3):
    timestamp = row['Timestamp (s)']
    citation  = row['Citation contexte (FR)']

    window = csv_df[(csv_df['t_start'] >= timestamp - window_back) &
                    (csv_df['t_start'] <= timestamp + window_fwd)].copy()
    if len(window) == 0:
        return None

    window['overlap'] = window['whisper_fr'].apply(
        lambda x: content_overlap_ratio(citation, x))

    if window['overlap'].max() == 0:
        return None

    anchor_idx = window['overlap'].idxmax()
    anchor_pos = window.index.get_loc(anchor_idx)
    extended = [anchor_idx]

    for offset in range(1, max_extension + 1):
        next_pos = anchor_pos + offset
        if next_pos < len(window) and window.iloc[next_pos]['overlap'] > 0:
            extended.append(window.index[next_pos])
        else:
            break

    return csv_df.loc[extended].sort_values('t_start')

## 6. Exécution de l'alignement

In [ ]:
# Save dislocation_translations.csv (carries the CALQUE and IDIOMATIC references to scoring).

rows = []
for _, drow in disloc.iterrows():
    matched = align_dislocation(drow, csv)

    base = {
        'ID': drow['ID'],
        'Speaker': drow['Speaker'],
        'Type': drow['Type'],
        'Citation contexte (FR)': drow['Citation contexte (FR)'],
        'Référence trad. CALQUE (EN)': drow['Référence trad. CALQUE (EN)'],
        'Référence trad. IDIOMATIC (EN)': drow['Référence trad. IDIOMATIC (EN)'],
    }

    if matched is None or len(matched) == 0:
        rows.append({**base,
            'n_segs': 0, 'segments': '', 'audio_start_s': None,
            'whisper_fr': '', 'xlsr_fr': '',
            'whisper_en_per_seg': '', 'whisper_en_cont': '',
            'xlsr_en': ''})
        continue

    rows.append({**base,
        'n_segs': len(matched),
        'segments': ' + '.join(matched['segment_id'].tolist()),
        'audio_start_s': round(float(matched.iloc[0]['t_start']), 2),
        'whisper_fr': ' '.join(' '.join(str(x).split()) for x in matched['whisper_fr'].fillna('')),
        'xlsr_fr':    ' '.join(' '.join(str(x).split()) for x in matched['xlsr_fr'].fillna('')),
        'whisper_en_per_seg': ' '.join(' '.join(str(x).split()) for x in matched['whisper_en_per_seg'].fillna('')),
        'whisper_en_cont': ' '.join(' '.join(str(x).split()) for x in matched['whisper_en_cont'].fillna('')),
        'xlsr_en':    ' '.join(' '.join(str(x).split()) for x in matched['xlsr_en'].fillna('')),
    })

aligned = pd.DataFrame(rows)

print(f'Dislocations traitées : {len(aligned)}')
print(f'  - avec >= 1 segment : {(aligned["n_segs"] > 0).sum()}')
print(f'  - sans match        : {(aligned["n_segs"] == 0).sum()}')
print(f'\nNb segments par disloc : min={aligned["n_segs"].min()}, '
      f'max={aligned["n_segs"].max()}, mean={aligned["n_segs"].mean():.1f}')

## 7. Sauvegarde du CSV

In [ ]:
# Quick preview for manual verification of anchors and spans.

aligned.to_csv(OUTPUT_PATH, index=False, sep=',', encoding='utf-8')
print(f'CSV écrit : {OUTPUT_PATH}')
print(f'Colonnes ({len(aligned.columns)}) : {list(aligned.columns)}')

## 8. Aperçu

Inspection rapide des premières lignes pour vérification.

In [ ]:
# (step continued)

pd.set_option('display.max_colwidth', 80)
aligned[['ID', 'Speaker', 'Type', 'audio_start_s', 'n_segs', 'segments',
         'Citation contexte (FR)', 'whisper_fr']].head(10)